# 12 — Validate the MDRO mart

CRE and MDR *Pseudomonas aeruginosa* are measured separately from the
conventional first-culture susceptibility mart. `mart_mdro` reads **all
eligible culture observations** from `int_reporting_culture_organism`, not
phenotype-specific first-isolate records.

One patient contributes once per source release, cohort, base organism and
MDRO definition. Across eligible observations the status precedence is
**positive > not_evaluable > not_detected**. `phenotype_pct` counts positive
patients over **all eligible patients**, while `evaluable_pct` counts
determinate positive-or-not-detected patients over the same denominator.

### 1. Structural and arithmetic checks

Expected: unique keys; zero arithmetic, bounds, and missing-Gram errors.


In [3]:
%sql
select
    count(*) as mart_rows,
    count(distinct mdro_mart_key) as unique_keys,
    sum(case
        when denominator_count <> positive_count
                                + not_detected_count
                                + not_evaluable_count
          or evaluable_count <> positive_count + not_detected_count
            then 1 else 0
    end) as count_identity_mismatches,
    sum(case
        when positive_count < 0
          or not_detected_count < 0
          or not_evaluable_count < 0
          or positive_count > denominator_count
            then 1 else 0
    end) as invalid_counts,
    sum(case
        when phenotype_pct <> round(
            100.0 * positive_count / nullif(denominator_count, 0), 1
        ) or evaluable_pct <> round(
            100.0 * evaluable_count / nullif(denominator_count, 0), 1
        ) then 1 else 0
    end) as percentage_formula_mismatches,
    sum(case
        when phenotype_pct < 0 or phenotype_pct > 100
          or evaluable_pct < 0 or evaluable_pct > 100
            then 1 else 0
    end) as out_of_bounds_percentages,
    sum(case when gram_stain_group is null then 1 else 0 end)
        as missing_gram_classification
from workspace.micro_dev.mart_mdro;


,mart_rows,unique_keys,count_identity_mismatches,invalid_counts,percentage_formula_mismatches,out_of_bounds_percentages,missing_gram_classification
0,34,34,0,0,0,0,0


### 2. Independently reconstruct the patient-level MDRO states

This query reshapes CRE and MDR *P. aeruginosa* evidence into a common
observation-level table. For each applicable definition, it first combines
**all eligible observations for the same patient**, then aggregates the
patient statuses to the same release/cohort/organism/phenotype grain as the
mart. A full outer join detects both incorrect counts **and missing rows**.

The retired phenotype-first dbt model is intentionally not used.
Expect only `pass` rows in the result.


In [ ]:
%sql
with eligible_observations as (
    select
        source_version,
        cohort_id,
        patient_key,
        organism_reporting_group as organism_base_name,
        'CRE' as mdro_phenotype,
        ast_cre_status as phenotype_status
    from workspace.micro_dev.int_reporting_culture_organism
    where culture_time_shifted is not null
      and ast_cre_status <> 'not_applicable'

    union all

    select
        source_version,
        cohort_id,
        patient_key,
        organism_reporting_group as organism_base_name,
        'MDR Pseudomonas aeruginosa' as mdro_phenotype,
        ast_mdr_p_aeruginosa_status as phenotype_status
    from workspace.micro_dev.int_reporting_culture_organism
    where culture_time_shifted is not null
      and ast_mdr_p_aeruginosa_status <> 'not_applicable'
),
patient_evidence as (
    select
        source_version,
        cohort_id,
        patient_key,
        organism_base_name,
        mdro_phenotype,
        max(case when phenotype_status = 'positive' then 1 else 0 end)
            as any_positive,
        max(case when phenotype_status = 'not_evaluable' then 1 else 0 end)
            as any_not_evaluable,
        max(case when phenotype_status = 'not_detected' then 1 else 0 end)
            as any_not_detected
    from eligible_observations
    group by
        source_version,
        cohort_id,
        patient_key,
        organism_base_name,
        mdro_phenotype
),
patient_status as (
    select
        source_version,
        cohort_id,
        patient_key,
        organism_base_name,
        mdro_phenotype,
        case
            when any_positive = 1 then 'positive'
            when any_not_evaluable = 1 then 'not_evaluable'
            when any_not_detected = 1 then 'not_detected'
            else null
        end as expected_status
    from patient_evidence
),
expected_summary as (
    select
        source_version,
        cohort_id,
        organism_base_name,
        mdro_phenotype,
        count(*) as expected_denominator,
        sum(case when expected_status = 'positive' then 1 else 0 end)
            as expected_positive,
        sum(case when expected_status = 'not_detected' then 1 else 0 end)
            as expected_not_detected,
        sum(case when expected_status = 'not_evaluable' then 1 else 0 end)
            as expected_not_evaluable,
        sum(case when expected_status is null then 1 else 0 end)
            as unexpected_null_patient_status
    from patient_status
    group by
        source_version,
        cohort_id,
        organism_base_name,
        mdro_phenotype
),
comparison as (
    select
        coalesce(e.source_version, m.source_version) as source_version,
        coalesce(e.cohort_id, m.cohort_id) as cohort_id,
        coalesce(e.organism_base_name, m.organism_base_name)
            as organism_base_name,
        coalesce(e.mdro_phenotype, m.mdro_phenotype) as mdro_phenotype,
        case
            when e.source_version is null or m.mdro_mart_key is null
              or e.unexpected_null_patient_status <> 0
              or e.expected_denominator <> m.denominator_count
              or e.expected_positive <> m.positive_count
              or e.expected_not_detected <> m.not_detected_count
              or e.expected_not_evaluable <> m.not_evaluable_count
                then 'mismatch'
            else 'pass'
        end as validation_status
    from expected_summary e
    full outer join workspace.micro_dev.mart_mdro m
        on e.source_version = m.source_version
       and e.cohort_id = m.cohort_id
       and e.organism_base_name = m.organism_base_name
       and e.mdro_phenotype = m.mdro_phenotype
)
select
    validation_status,
    count(*) as mdro_mart_rows
from comparison
group by validation_status
order by validation_status;


The expected outcome is a single `pass` group covering every row in
`mart_mdro`. This validates the denominator and **all three** patient
status counts, including patients with incomplete AST evidence. A single
first resistant culture is not an appropriate independent numerator for
this metric.

### 3. Gram classification of MDRO reporting groups


In [ ]:
%sql
select
    gram_stain_group,
    count(distinct organism_base_name) as organism_count,
    count(*) as mart_rows

from workspace.micro_dev.mart_mdro

group by gram_stain_group

order by gram_stain_group;

CRE and MDR *P. aeruginosa* apply to Gram-negative organisms in this
project's phenotype definitions. Verify the current counts in the query
output rather than relying on a saved number from an older build.

### 4. Inspect organism and phenotype breakdown


In [ ]:
%sql
select
    organism_base_name,
    gram_stain_group,
    mdro_phenotype,
    count(*) as mart_rows

from workspace.micro_dev.mart_mdro

group by
    organism_base_name,
    gram_stain_group,
    mdro_phenotype

order by
    gram_stain_group,
    organism_base_name,
    mdro_phenotype;

Some organisms may have MDRO summaries even when they have no approved
organism–antibiotic combinations in the conventional susceptibility mart.

### 5. Confirm the Gram reference covers both marts


In [ ]:
%sql
with reporting_organisms as (

    select distinct
        organism_reporting_group
    from workspace.micro_dev.mart_susceptibility

    union

    select distinct
        organism_base_name as organism_reporting_group
    from workspace.micro_dev.mart_mdro

)

select
    r.organism_reporting_group

from reporting_organisms r

left join workspace.micro_dev.organism_reporting_groups g
    on r.organism_reporting_group = g.organism_reporting_group

where g.organism_reporting_group is null

order by r.organism_reporting_group;

Every reporting organism appearing in either mart should match the
`organism_reporting_groups` reference seed. A passing query returns zero
rows.

### 6. Check final-mart completeness


In [ ]:
%sql
select
    'mart_susceptibility' as model_name,
    count(*) as row_count,
    sum(
        case
            when gram_stain_group is null then 1
            else 0
        end
    ) as missing_gram_stain_group

from workspace.micro_dev.mart_susceptibility

union all

select
    'mart_mdro' as model_name,
    count(*) as row_count,
    sum(
        case
            when gram_stain_group is null then 1
            else 0
        end
    ) as missing_gram_stain_group

from workspace.micro_dev.mart_mdro;

The latest validated build reported **455 susceptibility-mart rows** and
**31 MDRO-mart rows**, with complete Gram classification. The query above
should confirm counts after the cleanup; prior saved notebook outputs are
not reused as evidence of the revised patient-level reconstruction.

### Conclusion

The MDRO mart uses an all-observations, patient-level evidence policy,
whereas the susceptibility mart uses the first overall culture per
patient/organism/cohort. Their denominators answer different questions
and must not be combined. After rerunning these checks, the two validated
reporting marts can feed their separate Tableau views.
